# Binding prediction and aptamer generation using AptaMCTS
Step-by-step guide for using AptaMCTS for aptamer-protein binding prediction and
for proposing new aptamers for a target protein.

## Overview
AptaMCTS predicts whether an aptamer and a protein interact from the two
sequences alone, and uses that prediction as the score of a Monte Carlo tree
search that proposes new aptamers. The classes used in this notebook:

- **MoleculeLoader**: lazy container for aptamer/protein data; cells may hold
  in-memory sequences or file paths (PDB/FASTA/FASTQ), materialized to a
  DataFrame via `to_dataframe()`.
- **AptaMCTSPipeline**: the end-to-end estimator. It converts the aptamer to
  RNA with `DNAToRNA`, encodes it with `KMerFrequencies` (k-mers of length 1
  to 4 over `ACGU`, divided by the sequence length), encodes the protein with
  `ConjointTriad` (k-mers of length 1 to 3 over the 7 conjoint triad groups),
  and classifies the pair with a random forest.
- **AptamerEvalAptaMCTS**: scores a candidate aptamer against one target
  protein with a fitted pipeline.
- **MCTS**: the tree search that builds a candidate one nucleotide at a time
  and keeps the best scored sequence.

## Data preparation
The notebook uses the li2014 dataset from the AptaMCTS paper, bundled in
`pyaptamer/datasets/data/train_li2014.csv` and `test_li2014.csv`. Each row is an
aptamer sequence, a protein sequence and the label `positive` or `negative`.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from pyaptamer.datasets import load_li2014

X_train, y_train = load_li2014(split="train", return_X_y=True)
X_test, y_test = load_li2014(split="test", return_X_y=True)

In [ ]:
# Display the first few aptamer-protein pairs (materialized from the loader)
X_train.to_dataframe().head()

In [ ]:
# Display the first few labels
y_train.head()

## Fitting the pipeline
`AptaMCTSPipeline` takes a `MoleculeLoader` or a `DataFrame` with an `aptamer`
and a `protein` column. The aptamers in li2014 are written as DNA and as RNA;
the pipeline converts both to RNA before encoding.

In [ ]:
from pyaptamer.aptamcts import AptaMCTSPipeline

pipeline = AptaMCTSPipeline()
pipeline.fit(X_train, y_train["label"])

# Predict the labels for the test split
y_pred = pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, matthews_corrcoef

print("Test accuracy:", accuracy_score(y_test["label"], y_pred))
print("Test MCC:", matthews_corrcoef(y_test["label"], y_pred))

In [ ]:
# Class probabilities, columns in the order of pipeline.pipeline_.classes_
pipeline.predict_proba(X_test)[:5]

## Feature extraction
The pipeline runs three transformers, each on one column with `ApplyToCols`.
The same features can be computed on their own, for example to feed a
different classifier.

In [ ]:
from sklearn.pipeline import Pipeline

from pyaptamer.transformations.compose import ApplyToCols
from pyaptamer.transformations.encode import ConjointTriad, KMerFrequencies
from pyaptamer.transformations.transform import DNAToRNA

features = Pipeline(
    [
        ("to_rna", ApplyToCols(DNAToRNA(), cols="aptamer")),
        (
            "aptamer",
            ApplyToCols(
                KMerFrequencies(k=4, alphabet="ACGU", normalize="length"),
                cols="aptamer",
            ),
        ),
        ("protein", ApplyToCols(ConjointTriad(k=3), cols="protein")),
    ]
)

Xt = features.fit_transform(X_train)
Xt.shape

In [ ]:
# 340 aptamer columns followed by 399 protein columns
Xt.iloc[:3, [0, 1, 2, 340, 341, 342]]

## Swapping the estimator
The default estimator is `RandomForestClassifier(class_weight="balanced")`.
The paper trained forests of 35 to 200 trees and kept the best one. Any
scikit-learn classifier can take its place.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=200, class_weight="balanced")
pipeline = AptaMCTSPipeline(estimator=forest)
pipeline.fit(X_train, y_train["label"])

print("Test accuracy:", accuracy_score(y_test["label"], pipeline.predict(X_test)))

# Using `AptaMCTS` on new sequences

## Binding probability for a protein and a candidate aptamer
The target protein comes from the 1GNH structure through the dataset loader.
With `tiling="first"` the loader returns the sequence of the first chain.

In [ ]:
import pandas as pd

from pyaptamer.data import MoleculeLoader
from pyaptamer.datasets import load_1gnh

target = load_1gnh(tiling="first").to_dataframe()["sequence"].iloc[0]
aptamer = "GGGAGGACGAAGACGACTCGAGACAGGCTAGGGAGGGA"

X_new = MoleculeLoader(data=pd.DataFrame({"aptamer": [aptamer], "protein": [target]}))

pipeline.predict_proba(X_new)

## Candidate aptamers from a FASTA file
`MoleculeLoader` parses the FASTA. With `tiling="samples"` it gives one row per
record and repeats the single target protein on every row.

In [ ]:
import tempfile
from pathlib import Path

candidate_aptamers = [
    "GGGAGGACGAAGACGACUCGAGACAGGCUAGGGAGGGA",
    "AAGCGUCGGAUCUACACGUGCGAUAGCUCAGUACGCGGU",
    "CGGUAUCGAGUACAGGAGUCCGACGGAUAGUCCGGAGC",
]
fasta_path = Path(tempfile.gettempdir()) / "candidate_aptamers.fasta"
with open(fasta_path, "w") as fh:
    for i, seq in enumerate(candidate_aptamers):
        fh.write(f">candidate{i}\n{seq}\n")

X_fasta = MoleculeLoader(
    data=pd.DataFrame({"aptamer": [fasta_path], "protein": [target]}),
    tiling="samples",
)

X_fasta.to_dataframe()

In [ ]:
pipeline.predict_proba(X_fasta)

## Proposing new aptamers with MCTS
`AptamerEvalAptaMCTS` scores every candidate the search proposes against the
target protein. `depth` is the length of the generated aptamer. The paper uses
many more iterations; 100 keeps this cell short.

In [ ]:
from pyaptamer.experiments import AptamerEvalAptaMCTS
from pyaptamer.mcts import MCTS

experiment = AptamerEvalAptaMCTS(target=target, pipeline=pipeline)

mcts = MCTS(experiment=experiment, depth=20, n_iterations=100)
output = mcts.run(verbose=False)

In [ ]:
# The best sequence found and its predicted binding probability
output["candidate"], output["score"]